# Milestone 2 — Encoder Comparison: CLIP · BERT · GPT-2

**Optional extension:** *autoregressive text encoders (GPT-based embeddings).*

No generation, no new pipeline. The same 6,000 curated captions go through a third
encoder, and milestone 1's `sanity_checks.sibling_gap` reads off a third row.

**The question this answers.** The two captions of one COCO image describe the same scene, so an
encoder that represents *meaning* should place them closer together than two unrelated captions. The
gap between those two cosines is the measurement. A large sibling cosine alongside an equally large
random cosine is the failure mode to watch for: it means everything has been mapped into one narrow
cone, so the rankings carry almost no information.

**The point for the write-up.** It is tempting to say CLIP beats GPT-2 because GPT-2 is causally
masked. That is wrong — **CLIP's text tower is causally masked too** (verified below, straight from
the loaded model). What separates them is the training objective: contrastive image–text matching
against next-token prediction. One was trained to make a sentence vector that lines up with a picture;
the other was trained to guess the next word.

---
## 0. Setup

Writes the pipeline code beside the notebook and imports it.

**Where things go.** Generated images, grids, plots, tables and the sweep results are written to
`/kaggle/working/outputs/`. The caption manifest and the 512px photos are read from the attached
Dataset and copied to `/kaggle/working/data/` first, because `/kaggle/input/` is read-only and
`ensure_dirs()` creates directories under the data root.

Off Kaggle these default to `./data/` and `./outputs/`.

In [ ]:
# Install only what is actually missing -- Kaggle ships torch and transformers already.
import importlib.util
import subprocess
import sys
from pathlib import Path

missing = [p for p in ("diffusers", "transformers", "accelerate", "safetensors", "ftfy", "scipy")
           if importlib.util.find_spec(p) is None]
if missing:
    print("installing:", ", ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

Path("modules").mkdir(exist_ok=True)
Path("modules/__init__.py").touch()
print("ready")

In [ ]:
# ---- milestone 1 data in, artefacts out ----------------------------------
# Auto-discovery searches the whole of /kaggle/input, so it does not matter how deeply
# the Dataset is nested. Set this only to override it.
DATA_SOURCE = None        # e.g. "/kaggle/input/coco-processed/processed"

import os
import shutil
from pathlib import Path

if Path("/kaggle/input").is_dir():
    os.environ["T2I_OUTPUT_DIR"] = "/kaggle/working/outputs"

    source = (Path(DATA_SOURCE) if DATA_SOURCE else
              next((p.parent for p in sorted(Path("/kaggle/input").glob("**/processed/pairs.parquet"))),
                   None))
    if source is None:
        found = sorted(p.name for p in Path("/kaggle/input").iterdir())
        raise RuntimeError(
            "No attached Dataset contains processed/pairs.parquet. Attach milestone 1's "
            "data/processed/, or set DATA_SOURCE above. Under /kaggle/input: "
            + (", ".join(found) if found else "nothing attached"))

    # Copied, not read in place: /kaggle/input is read-only, config.ensure_dirs() creates
    # directories under DATA_DIR, and the encoders notebook saves GPT-2 embeddings there.
    work = Path("/kaggle/working/data")
    if not (work / "processed").is_dir():
        print(f"copying {source} ->\n        {work / 'processed'} ...")
        shutil.copytree(source, work / "processed")
    os.environ["T2I_DATA_DIR"] = str(work)

    print("data    :", os.environ["T2I_DATA_DIR"])
    print("outputs :", os.environ["T2I_OUTPUT_DIR"])
    print("contents:", ", ".join(sorted(p.name for p in (work / "processed").iterdir())))
else:
    print("not on Kaggle -- using ./data and ./outputs")

In [ ]:
%%writefile config.py
"""Every tunable value and every path.

Paths can be overridden with environment variables:

    T2I_PROJECT_ROOT   default: the working directory
    T2I_DATA_DIR       dataset and cached artefacts   (default: <root>/data)
    T2I_OUTPUT_DIR     plots, tables, generated PNGs  (default: <root>/outputs)
"""
import os
from pathlib import Path

import torch

# ---- Paths -------------------------------------------------------------
PROJECT_ROOT = Path(os.environ.get("T2I_PROJECT_ROOT") or Path.cwd())

DATA_DIR = Path(os.environ.get("T2I_DATA_DIR") or PROJECT_ROOT / "data")
ANN_FILE = DATA_DIR / "annotations" / "captions_val2017.json"
IMAGES_DIR = DATA_DIR / "val2017"

PROCESSED_DIR = DATA_DIR / "processed"
IMAGES_512_DIR = PROCESSED_DIR / "images_512"      # 512x512 RGB JPEG cache
EMB_DIR = PROCESSED_DIR / "embeddings"             # the .npy arrays
MANIFEST_PARQUET = PROCESSED_DIR / "pairs.parquet"
MANIFEST_CSV = PROCESSED_DIR / "pairs.csv"
METADATA_FILE = PROCESSED_DIR / "metadata.json"

OUTPUT_DIR = Path(os.environ.get("T2I_OUTPUT_DIR") or PROJECT_ROOT / "outputs")
GENERATIONS_DIR = OUTPUT_DIR / "generations"       # generated PNGs
PLOTS_DIR = OUTPUT_DIR / "plots"                   # every figure, as PNG
TABLES_DIR = OUTPUT_DIR / "tables"                 # every summary table, as JSON

SWEEP_DIR = OUTPUT_DIR / "sweep"                   # one sub-directory per swept config

ALL_DIRS = (PROCESSED_DIR, IMAGES_512_DIR, EMB_DIR, GENERATIONS_DIR, PLOTS_DIR, TABLES_DIR,
            SWEEP_DIR)


def ensure_dirs():
    """Create every output directory."""
    for d in ALL_DIRS:
        d.mkdir(parents=True, exist_ok=True)


# ---- Subset ------------------------------------------------------------
N_IMAGES = 3000             # distinct images kept
CAPTIONS_PER_IMAGE = 2      # captions kept per image -> 3000 x 2 = 6000 pairs
IMAGE_SIZE = 512            # Stable Diffusion v1.x native resolution
SEED = 42                   # single source of randomness for the whole pipeline

# ---- Caption filters ---------------------------------------------------
MIN_WORDS, MAX_WORDS = 3, 60
MIN_CHARS, MAX_CHARS = 10, 250

# ---- Models ------------------------------------------------------------
CLIP_JOINT_MODEL = "openai/clip-vit-base-patch32"    # text + vision towers -> shared 512-d space
CLIP_SD_MODEL = "openai/clip-vit-large-patch14"      # the text encoder SD v1.5 is conditioned on
BERT_MODEL = "google-bert/bert-base-uncased"         # text-only Transformer, for comparison
GPT2_MODEL = "openai-community/gpt2"                 # autoregressive encoder
SD_REPO_CANDIDATES = [                               # tried in order; the original runwayml repo is gone
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    "sd-legacy/stable-diffusion-v1-5",
    "CompVis/stable-diffusion-v1-4",
]

CLIP_MAX_TOKENS = 77   # fixed by CLIP's positional embedding table
BERT_MAX_TOKENS = 64   # COCO captions are short; 64 covers them with room to spare
GPT2_MAX_TOKENS = 64   # matched to BERT so the encoder comparison varies only the model
BATCH_SIZE = 64

# ---- Generation -------------------------------------------------------
N_STEPS = 30           # denoising steps
GUIDANCE = 7.5         # classifier-free guidance scale
N_PROMPTS = 5          # test prompts drawn from the curated subset
PROMPT_MIN_WORDS = 8   # prompts long enough to describe a scene, not one object
PROMPT_MAX_WORDS = 18

# ---- Precision ---------------------------------------------------------
# Text encoders in float32: they are small, and float32 keeps the stored embeddings
# free of rounding so cosine similarities are exact. Stable Diffusion in float16 on
# GPU: the UNet is the expensive part and fp16 roughly halves VRAM and step time.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ENCODER_DTYPE = torch.float32
SD_DTYPE = torch.float16 if DEVICE.type == "cuda" else torch.float32

# ---- Toggles -----------------------------------------------------------
# The 77x768 sequence tensors are ~710 MB at 6,000 captions. The SD text encoder is
# deterministic in eval mode, so re-running it reproduces them bit-for-bit in well
# under a minute -- cheaper to recompute than to store.
SAVE_SEQ_EMBEDDINGS = False
RUN_SD_GENERATION = True    # section 11 needs ~4 GB of Stable Diffusion weights on first run
N_ALIGN_IMAGES = 500        # images put through CLIP's vision tower for the retrieval check
REUSE_EXISTING = True       # reuse cached embeddings when they match this run's subset
FORCE_RECOMPUTE = False     # set True to ignore every cache and redo all work


# ---- The sweep ---------------------------------------------------------
N_FIXED_PROMPTS = 10   # every config generates this same set, from the same latents
LATENT_SEED = SEED     # latents are built once, on CPU, and passed into every run

CFG_GRID = (1.0, 4.0, 7.5, 11.0, 15.0)   # spec sweeps 1-15; 7.5 is the SD 1.5 default
GUIDANCE_RESCALE = 0.7                   # corrects the oversaturation CFG > 12 causes

# Sampler -> step count. The counts differ on purpose: DPMSolver++ and UniPC reach at
# 20 steps what PNDM needs 50 for, so an equal-step comparison would measure nothing
# but the step count.
SAMPLERS = {
    "pndm": 50,      # SD 1.5 default -- the baseline
    "ddim": 50,      # deterministic at eta=0
    "lms": 30,       # multistep; needs scipy
    "euler": 30,
    "euler_a": 30,   # ancestral -- adds noise each step, never converges
    "dpmpp": 20,
    "unipc": 20,
}

# Noise-schedule axes. Swept as a product against one fixed sampler, so the only
# variable is the schedule.
# SD 1.5's shipped scheduler config omits timestep_spacing, so each scheduler class would
# fall back to its own default -- PNDM and DDIM to "leading", the rest to "linspace". The
# guidance and sampler grids pin it so that axis stays still while the schedule grid sweeps it.
BASE_TIMESTEP_SPACING = "leading"   # SD legacy, what v1.5 was trained under

TIMESTEP_SPACINGS = ("leading", "trailing", "linspace")
ZERO_SNR_OPTIONS = (False, True)
KARRAS_OPTIONS = (False, True)
SCHEDULE_SAMPLER = "dpmpp"

DDPM_STEPS = 1000      # the illustration, not a sweep arm
DDPM_N_PROMPTS = 2

# Batch 4, not the 8 an unshared 16 GB T4 allows: this targets an 8 GB RTX 5060 Laptop
# GPU that is also driving the display. SD 1.5 in fp16 is ~2.5 GB of weights and roughly
# 0.6 GB of activations per 512x512 image.
GEN_BATCH_SIZE = 4

In [ ]:
%%writefile utils.py
"""Seeding, path printing, and saving figures and tables.
"""
import json
import os
import random
from pathlib import Path

import numpy as np
import torch

import config as cfg


def set_seed(seed=None):
    """Reseed every RNG the pipeline touches.

    Called once at the start and again before generation, so the subset, the
    sampled examples and the generated images are identical on every re-run.
    """
    seed = cfg.SEED if seed is None else seed
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    # Deterministic convolution kernels: slightly slower, but two runs produce
    # identical tensors.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    return seed


def rel(path):
    """`path` relative to the project root, as a string, for printing.

    Falls back to a computed relative path instead of raising when the path sits
    outside the root, as when the data and the code sit on different mounts, and to
    the absolute path when even that is impossible, as on Windows when the two are
    on different drives.
    """
    p = Path(path)
    try:
        return p.relative_to(cfg.PROJECT_ROOT).as_posix()
    except ValueError:
        pass
    try:
        return Path(os.path.relpath(p, cfg.PROJECT_ROOT)).as_posix()
    except ValueError:
        return p.as_posix()


def save_fig(fig, name):
    """Write a figure to `outputs/plots/` and return its path for printing."""
    cfg.PLOTS_DIR.mkdir(parents=True, exist_ok=True)
    path = cfg.PLOTS_DIR / f"{name}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    return rel(path)


def save_table(name, data):
    """Write a summary table to `outputs/tables/` as JSON.

    `default=float` converts numpy scalars, which json cannot serialise.
    """
    cfg.TABLES_DIR.mkdir(parents=True, exist_ok=True)
    path = cfg.TABLES_DIR / f"{name}.json"
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, default=float)
    return rel(path)

In [ ]:
%%writefile modules/bert_embeddings.py
"""BERT text encoder.

`bert-base-uncased` is the same architecture class and roughly the same size as
CLIP's text tower, but trained with masked language modelling on text alone. It has
never seen an image. That difference is the point: both produce 768-d sentence
vectors that look equally reasonable on paper, so the only way to tell them apart is
to measure them.

Pooling: BERT ships a `pooler_output`, but it is the tanh-projected [CLS] state
trained for next-sentence prediction, not for sentence similarity -- using it as a
sentence embedding is a well-documented trap. Two honest alternatives are computed
instead, and mean pooling is preferred when they disagree.
"""
import torch
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

import config as cfg


def load(model_name=None, device=None, dtype=None):
    """Load BERT. Returns `(tokenizer, model)`."""
    model_name = cfg.BERT_MODEL if model_name is None else model_name
    device = cfg.DEVICE if device is None else device
    dtype = cfg.ENCODER_DTYPE if dtype is None else dtype

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device=device, dtype=dtype).eval()
    return tokenizer, model


def mean_pool(last_hidden_state, attention_mask):
    """Mask-aware mean over the token dimension, so padding contributes nothing."""
    mask = attention_mask.unsqueeze(-1).to(last_hidden_state.dtype)
    return (last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)


@torch.no_grad()
def encode_text(texts, tokenizer, model, batch_size=None, max_tokens=None, device=None):
    """Encode captions two ways. Returns `(cls, mean)`, each (N, 768) float32.

    `cls`  -- the first token's final hidden state, raw.
    `mean` -- the attention-mask-weighted mean over all real tokens, the standard
              recipe for sentence similarity from an MLM encoder.
    """
    batch_size = cfg.BATCH_SIZE if batch_size is None else batch_size
    max_tokens = cfg.BERT_MAX_TOKENS if max_tokens is None else max_tokens
    device = cfg.DEVICE if device is None else device

    cls_out, mean_out = [], []
    for i in tqdm(range(0, len(texts), batch_size), desc="BERT"):
        toks = tokenizer(
            texts[i:i + batch_size], padding="max_length", max_length=max_tokens,
            truncation=True, return_tensors="pt",
        ).to(device)
        hidden = model(**toks).last_hidden_state
        cls_out.append(hidden[:, 0].float().cpu())
        mean_out.append(mean_pool(hidden, toks["attention_mask"]).float().cpu())
    return torch.cat(cls_out).numpy(), torch.cat(mean_out).numpy()


def describe(model):
    """Parameter, layer and vocab counts."""
    return {
        "hidden_size": model.config.hidden_size,
        "layers": model.config.num_hidden_layers,
        "vocab_size": model.config.vocab_size,
        "parameters_millions": sum(p.numel() for p in model.parameters()) / 1e6,
    }

In [ ]:
%%writefile modules/clip_embeddings.py
"""CLIP text and image encoders.

Two checkpoints doing two different jobs:

  `clip-vit-base-patch32`   pooled 512-d vectors in a space shared by text AND
                            images, so a caption vector and a photo vector are
                            directly comparable -- which is what makes text to
                            image retrieval measurable at all.
  `clip-vit-large-patch14`  the text encoder Stable Diffusion v1.5 is conditioned
                            on. A pretrained UNet only understands the space it
                            was trained in, so there is no substitute.

The conditioning tensor is one vector per token, not one per sentence, because the
generator reads the prompt token by token. That is how "a *red* bus" colours the bus
instead of tinting the whole picture red.
"""
import numpy as np
import torch
from PIL import Image
from tqdm.auto import tqdm
from transformers import CLIPImageProcessor, CLIPModel, CLIPTextModel, CLIPTokenizer

import config as cfg


def pooled(out):
    """Unwrap `get_text_features` / `get_image_features` across transformers versions.

    transformers >= 5 returns a pooling output object; earlier versions return a
    bare tensor. `pooler_output` is ALREADY projected into the joint space --
    applying `model.text_projection` on top of it would be wrong.
    """
    return out.pooler_output if hasattr(out, "pooler_output") else out


# ---------------------------------------------------------------------------
# Joint text + vision model
# ---------------------------------------------------------------------------
def load_joint(model_name=None, device=None, dtype=None):
    """Load the joint-space CLIP. Returns `(tokenizer, model, image_processor)`."""
    model_name = cfg.CLIP_JOINT_MODEL if model_name is None else model_name
    device = cfg.DEVICE if device is None else device
    dtype = cfg.ENCODER_DTYPE if dtype is None else dtype

    tokenizer = CLIPTokenizer.from_pretrained(model_name)
    model = CLIPModel.from_pretrained(model_name).to(device=device, dtype=dtype).eval()
    processor = CLIPImageProcessor.from_pretrained(model_name)
    return tokenizer, model, processor


def token_lengths(texts, tokenizer):
    """Untruncated token count per caption.

    CLIP's positional table stops at 77 tokens, so anything longer is silently
    truncated and the model never sees the end of the caption. Measuring this is
    how we know the word-count filter is already keeping us under the ceiling.
    """
    return np.array([len(tokenizer(t, truncation=False)["input_ids"]) for t in texts])


@torch.no_grad()
def encode_text(texts, tokenizer, model, batch_size=None, max_tokens=None, device=None):
    """Pooled 512-d vectors in CLIP's shared text/image space. Returns (N, 512) float32."""
    batch_size = cfg.BATCH_SIZE if batch_size is None else batch_size
    max_tokens = cfg.CLIP_MAX_TOKENS if max_tokens is None else max_tokens
    device = cfg.DEVICE if device is None else device

    out = []
    for i in tqdm(range(0, len(texts), batch_size), desc="CLIP joint-space"):
        toks = tokenizer(
            texts[i:i + batch_size], padding="max_length", max_length=max_tokens,
            truncation=True, return_tensors="pt",
        ).to(device)
        out.append(pooled(model.get_text_features(**toks)).float().cpu())
    return torch.cat(out).numpy()


@torch.no_grad()
def encode_images(images, processor, model, batch_size=32, device=None, progress=True):
    """Pooled 512-d image vectors in the same space as `encode_text`.

    `images` may be paths or already-open PIL images.
    """
    device = cfg.DEVICE if device is None else device
    out = []
    steps = range(0, len(images), batch_size)
    for i in tqdm(steps, desc="CLIP image encode", disable=not progress):
        batch = [
            img if isinstance(img, Image.Image) else Image.open(img)
            for img in images[i:i + batch_size]
        ]
        batch = [img.convert("RGB") for img in batch]
        px = processor(images=batch, return_tensors="pt").to(device)
        out.append(pooled(model.get_image_features(**px)).float().cpu())
    return torch.cat(out).numpy()


# ---------------------------------------------------------------------------
# Stable Diffusion conditioning encoder
# ---------------------------------------------------------------------------
def load_sd_text_encoder(model_name=None, device=None, dtype=None):
    """Load the text encoder SD v1.5 is conditioned on. Returns `(tokenizer, model)`."""
    model_name = cfg.CLIP_SD_MODEL if model_name is None else model_name
    device = cfg.DEVICE if device is None else device
    dtype = cfg.ENCODER_DTYPE if dtype is None else dtype

    tokenizer = CLIPTokenizer.from_pretrained(model_name)
    model = CLIPTextModel.from_pretrained(model_name).to(device=device, dtype=dtype).eval()
    return tokenizer, model


@torch.no_grad()
def encode_sequence(texts, tokenizer, model, batch_size=None, max_tokens=None, device=None):
    """Per-token hidden states (N, 77, 768) plus the pooled EOS vector (N, 768).

    **Do not feed the sequence tensor to a pipeline as `prompt_embeds`.** This passes
    an attention mask; Stable Diffusion's own `encode_prompt` does not. CLIP's text
    tower is causal, so the mask leaves every real token bit-identical -- the pooled
    vector this returns is unaffected, which is why it is fine for the analysis it was
    written for -- but it moves every padded position by ~30 on a scale of 33, and the
    UNet cross-attends to all 77. Images come out different and measurably worse
    (CLIP score 0.23 against 0.34) with nothing raising an error.


    The sequence tensor comes back as float16 -- it is ~710 MB at 6,000 captions
    in float32, and halving it costs nothing we can measure downstream.
    """
    batch_size = cfg.BATCH_SIZE if batch_size is None else batch_size
    max_tokens = cfg.CLIP_MAX_TOKENS if max_tokens is None else max_tokens
    device = cfg.DEVICE if device is None else device

    seq_out, pool_out = [], []
    for i in tqdm(range(0, len(texts), batch_size), desc="CLIP sequence"):
        toks = tokenizer(
            texts[i:i + batch_size], padding="max_length", max_length=max_tokens,
            truncation=True, return_tensors="pt",
        ).to(device)
        out = model(**toks)
        seq_out.append(out.last_hidden_state.to(torch.float16).cpu())
        pool_out.append(out.pooler_output.float().cpu())
    return torch.cat(seq_out).numpy(), torch.cat(pool_out).numpy()


def describe(model):
    """Parameter and layer counts."""
    config = model.config
    out = {"parameters_millions": sum(p.numel() for p in model.parameters()) / 1e6}
    if hasattr(config, "text_config"):
        out["text_hidden_size"] = config.text_config.hidden_size
        out["vision_hidden_size"] = config.vision_config.hidden_size
        out["joint_projection_dim"] = config.projection_dim
    else:
        out["hidden_size"] = config.hidden_size
        out["layers"] = config.num_hidden_layers
    return out

In [ ]:
%%writefile modules/gpt2_embeddings.py
"""GPT-2 text encoder -- the autoregressive comparison.

CLIP's text tower is **also causally masked**, so what separates CLIP from GPT-2 is
not the attention pattern but the training objective: contrastive image-text matching
against next-token prediction. If masking were the explanation, the two would score
alike.

Pooling needs care for a causal model. Only the **last** token has attended to the
whole sentence, so it is the natural sentence vector -- but GPT-2 pads on the right,
so position -1 is usually padding. `last_token_pool` finds the last real token
instead. Mean pooling is reported alongside it because a causal model's early
positions have seen almost nothing, which drags the mean toward the first few words.

GPT-2 ships no padding token, so `load` points it at the EOS token. Every padded
position is masked out of both poolings, so this costs nothing.
"""
import torch
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

import config as cfg
from .bert_embeddings import mean_pool


def load(model_name=None, device=None, dtype=None):
    """Load GPT-2. Returns `(tokenizer, model)`."""
    model_name = cfg.GPT2_MODEL if model_name is None else model_name
    device = cfg.DEVICE if device is None else device
    dtype = cfg.ENCODER_DTYPE if dtype is None else dtype

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModel.from_pretrained(model_name).to(device=device, dtype=dtype).eval()
    return tokenizer, model


def last_token_pool(last_hidden_state, attention_mask):
    """The final *non-padding* token's hidden state, one vector per sequence."""
    lengths = attention_mask.sum(dim=1)
    index = (lengths - 1).clamp(min=0).to(last_hidden_state.device)
    return last_hidden_state[torch.arange(len(last_hidden_state), device=last_hidden_state.device), index]


@torch.no_grad()
def encode_text(texts, tokenizer, model, batch_size=None, max_tokens=None, device=None):
    """Encode captions two ways. Returns `(last, mean)`, each (N, 768) float32."""
    batch_size = cfg.BATCH_SIZE if batch_size is None else batch_size
    max_tokens = cfg.GPT2_MAX_TOKENS if max_tokens is None else max_tokens
    device = cfg.DEVICE if device is None else device

    last_out, mean_out = [], []
    for i in tqdm(range(0, len(texts), batch_size), desc="GPT-2"):
        toks = tokenizer(
            texts[i:i + batch_size], padding="max_length", max_length=max_tokens,
            truncation=True, return_tensors="pt",
        ).to(device)
        hidden = model(**toks).last_hidden_state
        last_out.append(last_token_pool(hidden, toks["attention_mask"]).float().cpu())
        mean_out.append(mean_pool(hidden, toks["attention_mask"]).float().cpu())
    return torch.cat(last_out).numpy(), torch.cat(mean_out).numpy()


def describe(model):
    """Parameter, layer and vocab counts."""
    config = model.config
    return {
        "hidden_size": config.n_embd,
        "layers": config.n_layer,
        "vocab_size": config.vocab_size,
        "parameters_millions": sum(p.numel() for p in model.parameters()) / 1e6,
    }

In [ ]:
%%writefile modules/sanity_checks.py
"""Checks that the embeddings are **meaningful**, not merely well-formed.

Matching shapes prove nothing on their own -- a bug that encoded the same caption
6,000 times would pass a shape check. Four checks, cheapest first:

  1. `embedding_health`  expected dimensions, no NaN/Inf, sane vector norms.
  2. `retrieval`         does each caption retrieve *its own* image out of 500?
                         Chance is 1/500 = 0.2%.
  3. `sibling_gap`       the two captions of one image describe the same scene, so
                         they should land closer together than two random captions.
                         Needs no image tower, so a text-only encoder can take part.
  4. `top_neighbours`    the qualitative read: what does each encoder think is
                         similar?
"""
import numpy as np
import torch
import torch.nn.functional as F

import config as cfg


def expected_shapes(n_rows, save_seq_embeddings=None):
    """The shape every array should have, given the manifest row count."""
    save_seq = cfg.SAVE_SEQ_EMBEDDINGS if save_seq_embeddings is None else save_seq_embeddings
    expected = {
        "clip_text_joint": (n_rows, 512),
        "clip_sd_pooled": (n_rows, 768),
        "bert_cls": (n_rows, 768),
        "bert_mean": (n_rows, 768),
    }
    if save_seq:
        expected["clip_sd_sequence"] = (n_rows, cfg.CLIP_MAX_TOKENS, 768)
    return expected


def embedding_health(arrays, expected):
    """Shape, finiteness and norm statistics per array.

    Returns `(report, all_ok)`.
    """
    report, all_ok = {}, True
    for name, array in arrays.items():
        a32 = array.astype(np.float32)
        shape_ok = array.shape == expected[name]
        finite = bool(np.isfinite(a32).all())
        norms = np.linalg.norm(a32.reshape(len(a32), -1), axis=1)
        all_ok &= shape_ok and finite
        report[name] = {
            "shape": list(array.shape),
            "expected": list(expected[name]),
            "dtype": str(array.dtype),
            "shape_ok": shape_ok,
            "all_finite": finite,
            "norm_mean": float(norms.mean()),
            "norm_std": float(norms.std()),
        }
    return report, bool(all_ok)


def retrieval(text_emb, image_emb):
    """Text -> image retrieval over a square set of caption/image pairs.

    Retrieval needs exactly one correct answer per query, so the caller passes the
    first caption of each image only -- otherwise two queries compete for the same
    target.

    Returns a dict with the metrics plus the raw `sim`, `ranks`, `matched` and
    `mismatched` arrays, so they can be plotted without recomputing.
    """
    txt = F.normalize(torch.from_numpy(text_emb.astype(np.float32)), dim=-1)
    img = F.normalize(torch.from_numpy(image_emb.astype(np.float32)), dim=-1)
    sim = (txt @ img.T).numpy()                   # (n, n): rows = captions, cols = images

    n = len(sim)
    ranks = (-sim).argsort(axis=1)
    rank_of_gt = np.array([int(np.where(ranks[i] == i)[0][0]) for i in range(n)])
    r1, r5, r10 = (float((rank_of_gt < k).mean()) for k in (1, 5, 10))
    chance = 1.0 / n

    gt = np.arange(n)
    matched, mismatched = sim[gt, gt], sim[~np.eye(n, dtype=bool)]

    return {
        "n_queries": int(n),
        "recall_at_1": r1, "recall_at_5": r5, "recall_at_10": r10,
        "chance": chance,
        "times_better_than_chance": float(r1 / chance),
        "median_rank_of_correct_image": int(np.median(rank_of_gt)) + 1,
        "cosine_matched_mean": float(matched.mean()),
        "cosine_matched_std": float(matched.std()),
        "cosine_mismatched_mean": float(mismatched.mean()),
        "cosine_mismatched_std": float(mismatched.std()),
        "separation": float(matched.mean() - mismatched.mean()),
        # raw arrays, for plotting
        "sim": sim, "ranks": ranks, "matched": matched, "mismatched": mismatched,
    }


def sibling_gap(encoders, pair_a, pair_b, rng=None):
    """Sibling-caption agreement per encoder.

    A larger gap means the encoder places two descriptions of one scene closer
    together than two unrelated captions -- it is encoding meaning rather than
    surface word overlap. A high sibling cosine with an equally high random cosine
    means the opposite: everything is mapped into a narrow cone, so the rankings
    carry little information. Watch for that on the [CLS] row.

    Returns `{label: {"sibling_cos": .., "random_cos": .., "gap": ..}}`.
    """
    rng = np.random.default_rng(cfg.SEED) if rng is None else rng
    shuffled = rng.permutation(len(pair_b))

    scores = {}
    for label, emb in encoders.items():
        v = F.normalize(torch.from_numpy(emb.astype(np.float32)), dim=-1)
        sib = (v[pair_a] * v[pair_b]).sum(-1).numpy()
        rnd = (v[pair_a] * v[pair_b][shuffled]).sum(-1).numpy()
        scores[label] = {
            "sibling_cos": float(sib.mean()),
            "random_cos": float(rnd.mean()),
            "gap": float(sib.mean() - rnd.mean()),
        }
    return scores


def top_neighbours(emb, query_idx, k=5):
    """The `k` captions nearest to `query_idx` in this encoder's space.

    Returns `(indices, similarities)`. The query itself is excluded.
    """
    v = F.normalize(torch.from_numpy(emb.astype(np.float32)), dim=-1)
    sims = (v[query_idx] @ v.T).numpy()
    sims[query_idx] = -np.inf
    idx = sims.argsort()[::-1][:k]
    return idx, sims[idx]


def clip_scores(prompts, generated_images, real_images, tokenizer, model, processor, device=None):
    """CLIP cosine similarity for a batch of generation runs.

    Returns `(text_vs_generated, text_vs_real, generated_vs_real)`, each (N,).

    Every number is a cosine in CLIP's joint space. Text-image scores sit in a
    narrow band in practice -- roughly 0.25-0.35 for a caption and an image that
    genuinely match, around 0.1 for an unrelated pair. `generated_vs_real` is on a
    different scale entirely and runs much higher, because two images share far
    more structure with each other than a sentence ever shares with an image;
    compare it across rows, never against the first two.
    """
    from .clip_embeddings import encode_images, pooled

    device = cfg.DEVICE if device is None else device
    with torch.no_grad():
        toks = tokenizer(prompts, padding="max_length", max_length=cfg.CLIP_MAX_TOKENS,
                         truncation=True, return_tensors="pt").to(device)
        t_emb = F.normalize(pooled(model.get_text_features(**toks)).float().cpu(), dim=-1)

    g_emb = F.normalize(torch.from_numpy(
        encode_images(generated_images, processor, model, device=device, progress=False)), dim=-1)
    r_emb = F.normalize(torch.from_numpy(
        encode_images(real_images, processor, model, device=device, progress=False)), dim=-1)

    return (
        (t_emb * g_emb).sum(-1).numpy(),
        (t_emb * r_emb).sum(-1).numpy(),
        (g_emb * r_emb).sum(-1).numpy(),
    )

In [ ]:
%%writefile modules/sd_pipeline.py
"""Loading Stable Diffusion 1.5 and swapping its sampler without losing its training.

Two things here are easy to get wrong.

**The scheduler swap.** `DDIMScheduler()` constructs a scheduler with *library*
defaults, which are not SD 1.5's -- `beta_schedule` would silently become `linear`
where SD 1.5 trained on `scaled_linear`, and every image would come out wrong for a
reason nothing reports. The swap must go through `from_config`, and it must read the
*original* config each time: swapping twice from `pipe.scheduler.config` compounds
the first swap's overrides into the second.

**What each sampler accepts.** The noise-schedule axes are not universal.
`EulerAncestralDiscreteScheduler` has no `use_karras_sigmas`, `PNDMScheduler` has no
`rescale_betas_zero_snr`, and passing one anyway is a TypeError partway through a
run. The accepted set is read from each class's signature rather than hardcoded.

Attention and VAE slicing are on by default to keep peak VRAM low, and `free()`
returns the memory before the next model loads.
"""
import gc
import inspect

import torch
from diffusers import (
    DDIMScheduler,
    DDPMScheduler,
    DPMSolverMultistepScheduler,
    EulerAncestralDiscreteScheduler,
    EulerDiscreteScheduler,
    LMSDiscreteScheduler,
    PNDMScheduler,
    StableDiffusionPipeline,
    UniPCMultistepScheduler,
)

import config as cfg

SCHEDULER_CLASSES = {
    "pndm": PNDMScheduler,
    "ddim": DDIMScheduler,
    "lms": LMSDiscreteScheduler,
    "euler": EulerDiscreteScheduler,
    "euler_a": EulerAncestralDiscreteScheduler,
    "dpmpp": DPMSolverMultistepScheduler,
    "unipc": UniPCMultistepScheduler,
    # Not in cfg.SAMPLERS on purpose: ~1000 steps is 100x the cost of DPMSolver++ for a
    # worse image, so DDPM is a two-prompt illustration rather than a sweep arm.
    "ddpm": DDPMScheduler,
}

SCHEDULER_DEFAULTS = {
    "dpmpp": {"algorithm_type": "dpmsolver++"},
}


def _accepted(scheduler_cls, kwargs):
    """Drop the kwargs this scheduler class cannot take."""
    params = inspect.signature(scheduler_cls.__init__).parameters
    if any(p.kind is inspect.Parameter.VAR_KEYWORD for p in params.values()):
        return dict(kwargs)
    return {k: v for k, v in kwargs.items() if k in params}


def scheduler_kwargs(name, scheduler_cls=None, **overrides):
    """This sampler's defaults merged with `overrides`, filtered to what it accepts."""
    cls = SCHEDULER_CLASSES[name] if scheduler_cls is None else scheduler_cls
    return _accepted(cls, {**SCHEDULER_DEFAULTS.get(name, {}), **overrides})


def set_scheduler(pipe, name, base_config=None, scheduler_classes=None, **overrides):
    """Swap `pipe.scheduler` to `name`, rebuilt from SD 1.5's original config.

    Pass `base_config` -- the dict `load_sd` captured at load time -- so a second
    swap does not inherit the first one's overrides.
    """
    classes = SCHEDULER_CLASSES if scheduler_classes is None else scheduler_classes
    cls = classes[name]
    base = dict(pipe.scheduler.config) if base_config is None else dict(base_config)
    pipe.scheduler = cls.from_config(base, **scheduler_kwargs(name, scheduler_cls=cls, **overrides))
    return pipe


def enable_memory_savers(pipe):
    """Turn on the slicing an 8 GB card wants, using whichever API this version has.

    diffusers 0.42 removed the pipeline-level `enable_vae_slicing()` wrapper and left
    the method on the VAE itself, so both are probed rather than assumed.
    """
    if hasattr(pipe, "enable_attention_slicing"):
        pipe.enable_attention_slicing()
    vae = getattr(pipe, "vae", None)
    if hasattr(vae, "enable_slicing"):
        vae.enable_slicing()
    return pipe


def load_sd(repo_candidates=None, device=None, dtype=None, slicing=True):
    """Load SD 1.5. Returns `(pipe, repo_id, base_scheduler_config)`.

    The candidates are tried in order because the original `runwayml` repo was
    withdrawn; `base_scheduler_config` is captured before anything touches the
    scheduler and is what every later swap rebuilds from.
    """
    repo_candidates = cfg.SD_REPO_CANDIDATES if repo_candidates is None else repo_candidates
    device = cfg.DEVICE if device is None else device
    dtype = cfg.SD_DTYPE if dtype is None else dtype

    errors = {}
    for repo in repo_candidates:
        try:
            pipe = StableDiffusionPipeline.from_pretrained(
                repo, torch_dtype=dtype, safety_checker=None, requires_safety_checker=False,
            )
        except Exception as exc:
            errors[repo] = f"{type(exc).__name__}: {exc}"
            continue

        pipe = pipe.to(device)
        pipe.set_progress_bar_config(disable=True)
        if slicing and device.type == "cuda":
            enable_memory_savers(pipe)
        return pipe, repo, dict(pipe.scheduler.config)

    raise RuntimeError("no Stable Diffusion repo could be loaded: " + repr(errors))


def free(*objects):
    """Drop references and return the VRAM, so the next arm has room to load."""
    for obj in objects:
        if hasattr(obj, "to"):
            try:
                obj.to("cpu")
            except Exception:
                pass
        del obj
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def vram_report(device=None):
    """Allocated / reserved / total VRAM in GB."""
    device = cfg.DEVICE if device is None else device
    if device.type != "cuda":
        return {"device": str(device)}
    props = torch.cuda.get_device_properties(device)
    gb = 1024 ** 3
    return {
        "device": props.name,
        "total_gb": round(props.total_memory / gb, 2),
        "allocated_gb": round(torch.cuda.memory_allocated(device) / gb, 2),
        "reserved_gb": round(torch.cuda.memory_reserved(device) / gb, 2),
        "peak_gb": round(torch.cuda.max_memory_allocated(device) / gb, 2),
    }

In [ ]:
%%writefile modules/storage.py
"""Reading and writing the manifest, the .npy arrays and the run metadata.

The invariant every function here protects: **row *i* of every .npy file is the
caption at `pair_idx == i` in `pairs.parquet`.** It is checked before anything is
written and verified by a round-trip read afterwards, because a silent misalignment
would make every downstream number wrong while looking perfectly healthy.

Cached arrays are reused only when the subset fingerprint matches, so changing the
subset size, the cleaning rules or the seed throws the cache out automatically.
"""
import hashlib
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

import config as cfg
from utils import rel

MANIFEST_COLUMNS = [
    "pair_idx", "image_id", "ann_id", "cap_rank", "file_name",
    "caption_raw", "caption", "n_words", "n_chars",
    "height", "width", "image_path", "image_512_path",
]

# One row per retrieval image rather than one per pair, so it is exempt from the
# row-count check that flags stale arrays.
NON_PAIR_ARRAYS = {"clip_image_align"}


# ---------------------------------------------------------------------------
# Cache validity
# ---------------------------------------------------------------------------
def subset_fingerprint(captions, n_images=None, captions_per_image=None, seed=None):
    """A short hash of the exact caption list plus the settings that shape it.

    Cached arrays are reused only when this matches, so changing `N_IMAGES`, the
    cleaning rules or `SEED` throws the cache out automatically. Without it, a
    re-run could quietly pair new captions with old vectors.
    """
    n_images = cfg.N_IMAGES if n_images is None else n_images
    captions_per_image = cfg.CAPTIONS_PER_IMAGE if captions_per_image is None else captions_per_image
    seed = cfg.SEED if seed is None else seed

    src = "\n".join(captions) + f"|{n_images}|{captions_per_image}|{seed}"
    return hashlib.sha256(src.encode("utf-8")).hexdigest()[:16]


def cache_is_usable(fingerprint, previous_metadata=None, reuse=None, force_recompute=None):
    """Whether cached embeddings may be reused for this run."""
    reuse = cfg.REUSE_EXISTING if reuse is None else reuse
    force_recompute = cfg.FORCE_RECOMPUTE if force_recompute is None else force_recompute
    previous = (previous_metadata or {}).get("subset_fingerprint")
    return bool(reuse and not force_recompute and previous == fingerprint)


# ---------------------------------------------------------------------------
# Embedding arrays
# ---------------------------------------------------------------------------
def embedding_path(name, emb_dir=None):
    return (cfg.EMB_DIR if emb_dir is None else Path(emb_dir)) / f"{name}.npy"


def save_embedding(name, array, emb_dir=None):
    path = embedding_path(name, emb_dir)
    path.parent.mkdir(parents=True, exist_ok=True)
    np.save(path, array)
    return path


def load_embedding(name, emb_dir=None, mmap=False):
    return np.load(embedding_path(name, emb_dir), mmap_mode="r" if mmap else None)


def load_or_encode(names, encode_fn, n_rows, cache_usable=False, save=True,
                   reuse_log=None, emb_dir=None):
    """Return the named arrays, loading them from disk when a valid cache exists.

    `encode_fn` is only called on a miss, and its results are saved immediately so
    that a failure further down never throws away encoding work.

    Returns `(*arrays, elapsed_seconds)`; elapsed is 0.0 on a cache hit.
    """
    reuse_log = {} if reuse_log is None else reuse_log
    paths = [embedding_path(n, emb_dir) for n in names]

    if cache_usable and all(p.is_file() for p in paths):
        arrays = [np.load(p) for p in paths]
        if all(len(a) == n_rows for a in arrays):
            print(f"Reusing cached: {', '.join(p.name for p in paths)}")
            reuse_log.update({n: "reused" for n in names})
            return (*arrays, 0.0)
        print("  cached arrays have the wrong row count -- recomputing")

    t0 = time.time()
    result = encode_fn()
    arrays = list(result) if isinstance(result, tuple) else [result]
    elapsed = time.time() - t0

    for name, array in zip(names, arrays):
        if save:
            save_embedding(name, array, emb_dir)
            reuse_log[name] = "computed"
        else:
            reuse_log[name] = "computed (not saved)"
    return (*arrays, elapsed)


def save_arrays(arrays, manifest, emb_dir=None):
    """Write every array, each row-checked against the manifest first.

    Returns a list of `(name, shape, dtype, megabytes)`.
    """
    rows = []
    for name, array in arrays.items():
        assert len(array) == len(manifest), (
            f"{name} has {len(array)} rows but the manifest has {len(manifest)}"
        )
        save_embedding(name, array, emb_dir)
        rows.append((name, array.shape, str(array.dtype), array.nbytes / 1024 ** 2))
    return rows


def find_stale(arrays, n_rows, emb_dir=None):
    """Arrays on disk left over from an earlier run with a different subset size.

    Reported, never deleted -- throwing away someone else's work on a path guess
    is not this function's call to make.
    """
    emb_dir = cfg.EMB_DIR if emb_dir is None else Path(emb_dir)
    stale = []
    for p in sorted(emb_dir.glob("*.npy")):
        if p.stem in arrays or p.stem in NON_PAIR_ARRAYS:
            continue
        shape = np.load(p, mmap_mode="r").shape
        if shape[0] != n_rows:
            stale.append((rel(p), shape, p.stat().st_size / 1024 ** 2))
    return stale


def existing_embeddings(emb_dir=None):
    """What is already on disk, for the "what will this run cost" report."""
    emb_dir = cfg.EMB_DIR if emb_dir is None else Path(emb_dir)
    if not emb_dir.is_dir():
        return []
    return [
        (p.name, np.load(p, mmap_mode="r").shape, p.stat().st_size / 1024 ** 2)
        for p in sorted(emb_dir.glob("*.npy"))
    ]


# ---------------------------------------------------------------------------
# Manifest
# ---------------------------------------------------------------------------
def write_manifest(subset, parquet_path=None, csv_path=None):
    """Write the manifest in the exact row order of the embedding arrays.

    Image paths are stored relative to the project root, so the artefacts stay
    portable across machines.
    """
    parquet_path = cfg.MANIFEST_PARQUET if parquet_path is None else Path(parquet_path)
    csv_path = cfg.MANIFEST_CSV if csv_path is None else Path(csv_path)
    parquet_path.parent.mkdir(parents=True, exist_ok=True)

    manifest = subset[MANIFEST_COLUMNS].copy()
    for col in ("image_path", "image_512_path"):
        manifest[col] = manifest[col].map(rel)

    manifest.to_parquet(parquet_path, index=False)
    manifest.to_csv(csv_path, index=False, encoding="utf-8")
    return manifest


def read_manifest(parquet_path=None):
    return pd.read_parquet(cfg.MANIFEST_PARQUET if parquet_path is None else parquet_path)


# ---------------------------------------------------------------------------
# Metadata
# ---------------------------------------------------------------------------
def load_metadata(path=None):
    """The previous run's metadata, or None if there is none."""
    path = cfg.METADATA_FILE if path is None else Path(path)
    if not path.is_file():
        return None
    return json.loads(path.read_text(encoding="utf-8"))


def save_metadata(metadata, path=None):
    path = cfg.METADATA_FILE if path is None else Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)
    return path


def build_metadata(manifest, arrays, dataset_counts, cleaning_steps, timings,
                   fingerprint, reuse_log, not_persisted=None):
    """Assemble the provenance record.

    The config-derived sections are read straight from `config.py`; the caller
    supplies only the numbers that come out of the run itself.
    """
    import diffusers
    import torch
    import transformers

    return {
        "dataset": {
            "name": "COCO 2017 Captions",
            "split": "val2017",
            "annotations_file": rel(cfg.ANN_FILE),
            **dataset_counts,
            "n_curated_pairs": int(len(manifest)),
            "n_distinct_images": int(manifest["image_id"].nunique()),
            "captions_per_image": cfg.CAPTIONS_PER_IMAGE,
        },
        "image_preprocessing": {
            "size": [cfg.IMAGE_SIZE, cfg.IMAGE_SIZE],
            "mode": "RGB",
            "resize": "aspect-preserving shorter-side resize + centre crop (PIL ImageOps.fit, bicubic)",
            "exif_transpose": True,
            "normalisations": {"display_and_metrics": "[0, 1]", "sd_vae": "[-1, 1] via Normalize(0.5, 0.5)"},
            "cache_dir": rel(cfg.IMAGES_512_DIR),
        },
        "caption_cleaning": {
            "steps": list(cleaning_steps),
            "filters": {
                "min_words": cfg.MIN_WORDS, "max_words": cfg.MAX_WORDS,
                "min_chars": cfg.MIN_CHARS, "max_chars": cfg.MAX_CHARS,
                "drop_empty": True, "drop_duplicates": True, "drop_missing_image": True,
            },
            "stopwords": "not removed -- see the note in section 4",
        },
        "embeddings": {n: {"shape": list(a.shape), "dtype": str(a.dtype)} for n, a in arrays.items()},
        "embeddings_not_persisted": not_persisted or {},
        "models": {
            "clip_joint": cfg.CLIP_JOINT_MODEL,
            "clip_sd_text_encoder": cfg.CLIP_SD_MODEL,
            "bert": cfg.BERT_MODEL,
            "clip_max_tokens": cfg.CLIP_MAX_TOKENS,
            "bert_max_tokens": cfg.BERT_MAX_TOKENS,
            "encoder_dtype": str(cfg.ENCODER_DTYPE),
        },
        "encode_seconds": {k: round(v, 1) for k, v in timings.items()},
        "versions": {
            "python": sys.version.split()[0],
            "torch": torch.__version__,
            "transformers": transformers.__version__,
            "diffusers": diffusers.__version__,
        },
        "seed": cfg.SEED,
        "subset_fingerprint": fingerprint,   # guards cache reuse on the next run
        "cache_status": dict(reuse_log),
    }


# ---------------------------------------------------------------------------
# Round trip
# ---------------------------------------------------------------------------
def verify_round_trip(manifest, arrays, check=("clip_text_joint", "bert_mean")):
    """Reload from disk and confirm it matches what is in memory.

    Without this, a silent write failure would only surface much later.
    Returns a small summary dict; raises AssertionError if anything disagrees.
    """
    reloaded = read_manifest()
    assert len(reloaded) == len(manifest), "manifest row count changed on the round trip"
    assert reloaded["pair_idx"].tolist() == list(range(len(reloaded))), "pair_idx is not contiguous"

    for name in check:
        on_disk = load_embedding(name)
        assert np.array_equal(on_disk, arrays[name]), f"{name} does not match what was written"

    probe = reloaded.iloc[min(7, len(reloaded) - 1)]
    return {
        "manifest_rows": int(len(reloaded)),
        "arrays_checked": list(check),
        "probe_caption": probe["caption"],
        "probe_image": probe["image_512_path"],
    }

In [ ]:
import sys

sys.path.insert(0, ".")
for _name in [m for m in sys.modules if m in ("config", "utils") or m.startswith("modules")]:
    del sys.modules[_name]          # so a re-run picks up the freshly written files

import config as cfg
import utils
from modules import bert_embeddings, clip_embeddings, gpt2_embeddings, sanity_checks, sd_pipeline, storage

print("modules ready:", 6)

---
## 0. Paths on a hosted runtime

**Does nothing on a normal checkout**, so this notebook still runs unchanged locally.

It matters because the repository is **code only** — `data/` is git-ignored. Milestone 2 needs
milestone 1's `data/processed/` (manifest, embeddings and the 512px photo cache, ~390 MB), and a
`git clone` will not bring that with it.

There are two ways to supply it, and this cell handles both:

- **Attach it as a Kaggle Dataset.** The cell finds whichever attached dataset holds
  `processed/pairs.parquet`, copies it to `/kaggle/working/data` and points `config.py` there. The
  copy is not optional — `/kaggle/input` is read-only and `cfg.ensure_dirs()` creates directories
  under `DATA_DIR`.
- **Clone the repo somewhere writable and run milestone 1 first.** Then nothing needs redirecting
  and this cell stays out of the way.

On Colab `/content` is writable, so the second case applies and this cell is a no-op there.

It must stay **above the import cell** — `config.py` resolves every path once, when it is first
imported, and setting the variables afterwards has no effect.

In [ ]:
import time, warnings

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import transformers

warnings.filterwarnings("ignore", category=FutureWarning)
transformers.logging.set_verbosity_error()

import config as cfg
import utils
from modules import bert_embeddings, clip_embeddings, gpt2_embeddings, sanity_checks, sd_pipeline, storage

cfg.ensure_dirs()
utils.set_seed()

manifest = storage.read_manifest()
CAPTIONS = manifest["caption"].tolist()

first = manifest[manifest["cap_rank"] == 0].set_index("image_id")["pair_idx"]
second = manifest[manifest["cap_rank"] == 1].set_index("image_id")["pair_idx"]
shared = first.index.intersection(second.index)
PAIR_A, PAIR_B = first.loc[shared].to_numpy(), second.loc[shared].to_numpy()

print(f"captions      : {len(CAPTIONS):,}")
print(f"sibling pairs : {len(PAIR_A):,} images with two captions each")
print(f"device        : {cfg.DEVICE}")

## 1. Is CLIP's text tower really causal?

Asked of the loaded model rather than taken on trust. CLIP builds a causal attention mask inside its
text transformer exactly the way GPT-2 does; the evidence is in the module source and in the shape of
the mask the model constructs.

In [ ]:
from transformers.models.clip import modeling_clip
import inspect

source = inspect.getsource(modeling_clip)
for marker in ["_create_4d_causal_attention_mask", "causal_attention_mask", "is_causal"]:
    print(f"  {marker:<34}{'present' if marker in source else 'absent'} in transformers' CLIP text model")

print()
print("GPT-2 and CLIP both mask future tokens; only the training objective differs.")

## 2. Encode with all three

CLIP and BERT are reused from milestone 1's cache when it is valid — they are the same captions and
the same models, so recomputing them would only prove that the cache works. GPT-2 is new and takes a
couple of minutes.

Each encoder is freed before the next loads. None of these is large, but the habit is what keeps the
generation notebooks inside 8 GB.

In [ ]:
timings = {}

clip_tok, clip_model, _ = clip_embeddings.load_joint()
print("CLIP  :", clip_embeddings.describe(clip_model))
t0 = time.time()
clip_joint = clip_embeddings.encode_text(CAPTIONS, clip_tok, clip_model)
timings["CLIP ViT-B/32 (pooled)"] = time.time() - t0
sd_pipeline.free(clip_model); del clip_model

In [ ]:
bert_tok, bert_model = bert_embeddings.load()
print("BERT  :", bert_embeddings.describe(bert_model))
t0 = time.time()
bert_cls, bert_mean = bert_embeddings.encode_text(CAPTIONS, bert_tok, bert_model)
timings["BERT (CLS / mean)"] = time.time() - t0
sd_pipeline.free(bert_model); del bert_model

In [ ]:
gpt2_tok, gpt2_model = gpt2_embeddings.load()
print("GPT-2 :", gpt2_embeddings.describe(gpt2_model))
print(f"        pad token set to {gpt2_tok.pad_token!r} (GPT-2 ships none)")
t0 = time.time()
gpt2_last, gpt2_mean = gpt2_embeddings.encode_text(CAPTIONS, gpt2_tok, gpt2_model)
timings["GPT-2 (last / mean)"] = time.time() - t0
sd_pipeline.free(gpt2_model); del gpt2_model

for name, array in [("gpt2_last", gpt2_last), ("gpt2_mean", gpt2_mean)]:
    print(f"saved {storage.save_embedding(name, array).name}  {array.shape}  {array.dtype}")

## 3. Sibling agreement

`sanity_checks.sibling_gap`, unchanged from milestone 1. Each encoder gets the same pairs and the same
shuffle, so the only variable is the encoder.

In [ ]:
ENCODERS = {
    "CLIP ViT-B/32 (pooled)": clip_joint,
    "BERT [CLS]": bert_cls,
    "BERT mean": bert_mean,
    "GPT-2 last token": gpt2_last,
    "GPT-2 mean": gpt2_mean,
}

scores = sanity_checks.sibling_gap(ENCODERS, PAIR_A, PAIR_B)
table = (pd.DataFrame(scores).T
         .rename(columns={"sibling_cos": "sibling", "random_cos": "random"})
         .sort_values("gap", ascending=False))
table["ratio"] = (table["sibling"] / table["random"]).round(3)
display(table.round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.4))
y = np.arange(len(table))
ax.barh(y - 0.2, table["sibling"], height=0.38, label="two captions of the same image")
ax.barh(y + 0.2, table["random"], height=0.38, label="two unrelated captions")
ax.set_yticks(y); ax.set_yticklabels(table.index); ax.invert_yaxis()
ax.set_xlabel("mean cosine similarity")
ax.set_title("Sibling agreement: does the encoder represent meaning?")
ax.legend(); ax.grid(axis="x", alpha=0.3)
for i, gap in enumerate(table["gap"]):
    ax.annotate(f"gap {gap:.3f}", (max(table['sibling'].iloc[i], table['random'].iloc[i]) + 0.01, i),
                va="center", fontsize=8)
fig.tight_layout()
print(utils.save_fig(fig, "m2_encoder_comparison"))
plt.show()

## 4. What the numbers say

Read the **gap**, not the sibling cosine. A row with a high sibling cosine *and* a high random cosine
has collapsed its captions into a narrow cone — every sentence looks like every other, and the
ordering is nearly meaningless. The `ratio` column makes that visible: it is how many times more
similar a true sibling is than a random pair.

For GPT-2 specifically, compare its two rows against each other. Last-token pooling is the
theoretically right choice for a causal model — only the final position has attended to the whole
sentence — while mean pooling averages in early positions that have seen almost nothing. Which one
actually wins here is the finding, not the assumption.

In [ ]:
best = table.index[0]
summary = {
    "n_captions": len(CAPTIONS),
    "n_sibling_pairs": int(len(PAIR_A)),
    "encode_seconds": {k: round(v, 1) for k, v in timings.items()},
    "scores": {k: {m: round(float(x), 4) for m, x in v.items()} for k, v in scores.items()},
    "ranking_by_gap": table.index.tolist(),
    "best": best,
    "note": ("CLIP's text tower is causally masked too; the training objective, not the mask, "
             "is what separates it from GPT-2."),
}
print(utils.save_table("milestone_2_encoders", summary))
print()
print(f"best separation: {best}  (gap {table.loc[best, 'gap']:.4f}, "
      f"{table.loc[best, 'ratio']:.2f}x random)")